# Synergy Statistics Fix — ANOVA + Post-hoc Tests

Replaces separate paired t-tests (notebook 10c) with proper omnibus test first,
then post-hoc comparisons with multiple comparison correction.

**Design:** Repeated-measures ANOVA (or Friedman if non-normal) on 5 edge categories:
Border, Internal, Random, Random FC-Matched, Random Cycle Internal.
Then post-hoc pairwise tests with FDR (Benjamini-Hochberg) correction.

Addresses R1 ("synergy interpretation feels speculative") and R2-Q5
("why separate t-tests rather than single model? multiple comparison corrections?")

In [ ]:
import sys; sys.path.append("..")
from topofinger import config
import numpy as np
import pandas as pd
import pingouin as pg
from topofinger.stats_compat import normalize_pingouin
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, friedmanchisquare
from statsmodels.stats.multitest import multipletests
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# === Load saved synergy metrics ===
metrics_df = pd.read_pickle(config.RESULTS + '/synergy_metrics.pck')
print(f"Loaded: {metrics_df.shape[0]} subjects × {metrics_df.shape[1]} metrics")
print(metrics_df.describe().round(4))

## 1. Check Normality Assumptions

In [ ]:
column_names = ['border', 'internal', 'random', 'random_fc_matched', 'random_cycle_internal']
labels = ['Border', 'Internal', 'Random', 'Random\n(FC-Matched)', 'Internal\nRandom Cycles']
label_map = dict(zip(column_names, labels))

print("Shapiro-Wilk normality tests:")
print("-" * 50)
normality_ok = True
for col in column_names:
    stat, p = shapiro(metrics_df[col].dropna())
    status = "NORMAL" if p > 0.05 else "NON-NORMAL"
    if p <= 0.05:
        normality_ok = False
    print(f"  {label_map[col]:25s}: W={stat:.4f}, p={p:.4f} → {status}")

print(f"\nAll normal? {normality_ok}")
print("→ Using repeated-measures ANOVA (if normal) or Friedman test (if non-normal)")

## 2. Omnibus Test: Repeated-Measures ANOVA / Friedman

In [ ]:
# Reshape to long format for pingouin
long_df = metrics_df[column_names].copy()
long_df['subject'] = range(len(long_df))
long_df = long_df.melt(id_vars='subject', var_name='category', value_name='synergy')
long_df['category'] = long_df['category'].map(label_map)

print("=" * 70)
print("OMNIBUS TEST")
print("=" * 70)

if normality_ok:
    # Repeated-measures ANOVA
    print("\n--- Repeated-Measures ANOVA (all groups normal) ---")
    aov = normalize_pingouin(pg.rm_anova(data=long_df, dv='synergy', within='category', subject='subject',
                      correction=True, detailed=True))
    print(aov.to_string())
    
    F_val = aov['F'].values[0]
    p_val = aov['p_unc'].values[0]
    eta2 = aov['np2'].values[0]
    eps = aov['eps'].values[0]
    
    print(f"\nF = {F_val:.2f}, p = {p_val:.2e}, η²_p = {eta2:.4f}")
    print(f"Greenhouse-Geisser ε = {eps:.4f}")
    if 'p_GG_corr' in aov.columns:
        p_gg = aov['p_GG_corr'].values[0]
        print(f"GG-corrected p = {p_gg:.2e}")
    
    omnibus_method = "Repeated-Measures ANOVA"
    omnibus_p = p_val
else:
    # Friedman test (non-parametric alternative)
    print("\n--- Friedman Test (non-normal data detected) ---")
    data_arrays = [metrics_df[col].values for col in column_names]
    chi2, p_friedman = friedmanchisquare(*data_arrays)
    
    print(f"Friedman χ² = {chi2:.2f}, p = {p_friedman:.2e}")
    
    # Also compute Kendall's W (effect size for Friedman)
    n = len(metrics_df)
    k = len(column_names)
    W = chi2 / (n * (k - 1))
    print(f"Kendall's W = {W:.4f} (effect size)")
    
    omnibus_method = "Friedman"
    omnibus_p = p_friedman

print(f"\n→ Omnibus test ({omnibus_method}): p = {omnibus_p:.2e}")
if omnibus_p < 0.05:
    print("→ SIGNIFICANT — proceeding to post-hoc pairwise comparisons")
else:
    print("→ NOT SIGNIFICANT — pairwise comparisons may not be warranted")

## 3. Post-hoc Pairwise Comparisons (all 10 pairs, FDR-corrected)

In [ ]:
print("=" * 70)
print("POST-HOC PAIRWISE COMPARISONS")
print("=" * 70)

# Use pingouin's pairwise_tests for clean output
if normality_ok:
    posthoc = normalize_pingouin(pg.pairwise_tests(data=long_df, dv='synergy', within='category',
                                 subject='subject', parametric=True,
                                 padjust='fdr_bh', return_desc=True))
else:
    posthoc = normalize_pingouin(pg.pairwise_tests(data=long_df, dv='synergy', within='category',
                                 subject='subject', parametric=False,
                                 padjust='fdr_bh', return_desc=True))

# Display key columns
display_cols = ['A', 'B', 'mean(A)', 'std(A)', 'mean(B)', 'std(B)', 'T', 'W_val', 'p_unc', 'p_corr', 'hedges']
available = [c for c in display_cols if c in posthoc.columns]
print(posthoc[available].to_string())

In [ ]:
# Also compute Cohen's d (effect sizes) for each pair
print("\n" + "=" * 70)
print("EFFECT SIZES (Cohen's d) — ALL PAIRWISE")
print("=" * 70)

from itertools import combinations

effect_rows = []
for i, j in combinations(range(len(column_names)), 2):
    col_a, col_b = column_names[i], column_names[j]
    d = pg.compute_effsize(metrics_df[col_a], metrics_df[col_b], eftype='cohen')
    effect_rows.append({
        'Group A': label_map[col_a],
        'Group B': label_map[col_b],
        "Cohen's d": round(d, 4),
        'Interpretation': 'large' if abs(d) > 0.8 else ('medium' if abs(d) > 0.5 else ('small' if abs(d) > 0.2 else 'negligible'))
    })

effect_df = pd.DataFrame(effect_rows)
print(effect_df.to_string(index=False))

## 4. Summary Table for Paper

In [ ]:
print("=" * 70)
print("SUMMARY TABLE FOR PAPER")
print("=" * 70)

# Descriptive stats
desc = metrics_df[column_names].describe().T
desc['label'] = [label_map[c] for c in desc.index]
desc = desc[['label', 'mean', 'std', '50%', '25%', '75%']]
desc.columns = ['Category', 'Mean', 'SD', 'Median', 'Q1', 'Q3']
print("\nDescriptive Statistics:")
print(desc.to_string(index=False, float_format='{:.4f}'.format))

# Key comparisons (Internal vs each other)
print(f"\nOmnibus: {omnibus_method}, p = {omnibus_p:.2e}")
print("\nPost-hoc (Internal vs. others, FDR-corrected):")
internal_label = label_map['internal']
for _, row in posthoc.iterrows():
    if row['A'] == internal_label or row['B'] == internal_label:
        other = row['B'] if row['A'] == internal_label else row['A']
        p_corr = row['p_corr'] if 'p_corr' in row else row['p_unc']
        stars = '***' if p_corr < 0.001 else ('**' if p_corr < 0.01 else ('*' if p_corr < 0.05 else 'ns'))
        print(f"  Internal vs {other}: p_FDR = {p_corr:.2e} {stars}")

## 5. Updated Figure 4C — Boxplot with ANOVA + Post-hoc

In [ ]:
FIG_PATH = config.FIGURES_RAW + "/synergy_stats_fix/"
import os; os.makedirs(FIG_PATH, exist_ok=True)

# Custom colors: Border = violet, Internal = red, controls = blue
custom_colors = ['purple', 'red', 'royalblue', 'royalblue', 'royalblue']
plot_data = [metrics_df[col].dropna().values for col in column_names]

fig, ax = plt.subplots(figsize=(12, 9))
mpl.rcParams["axes.spines.right"] = False
mpl.rcParams["axes.spines.top"] = False

# Boxplot
box = ax.boxplot(plot_data, tick_labels=labels, patch_artist=True, showfliers=False,
                 medianprops=dict(color="black", linewidth=1.5))

for patch, color in zip(box['boxes'], custom_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.4)
    patch.set_edgecolor('black')
    patch.set_linewidth(1)

# Stripplot overlay 
for i, (data, color) in enumerate(zip(plot_data, custom_colors)):
    x_pos = np.random.normal(i + 1, 0.04, len(data))
    ax.scatter(x_pos, data, color=color, alpha=0.6, s=20, edgecolors='white', linewidth=0.5)

# Significance annotations (Internal vs each other, FDR-corrected)
internal_pos = 2  # 1-indexed position of Internal
y_max = max([d.max() for d in plot_data])
y_range = y_max - min([d.min() for d in plot_data])
annotation_height = y_max + 0.05 * y_range
height_increment = 0.12 * y_range

annotation_count = 0
internal_label = label_map['internal']
for _, row in posthoc.iterrows():
    if row['A'] == internal_label or row['B'] == internal_label:
        other = row['B'] if row['A'] == internal_label else row['A']
        other_pos = labels.index(other) + 1
        p_corr = row['p_corr'] if 'p_corr' in row else row['p_unc']
        
        current_height = annotation_height + annotation_count * height_increment
        
        # Bracket
        ax.plot([internal_pos, other_pos], [current_height, current_height], 'k-', linewidth=1)
        ax.plot([internal_pos, internal_pos], [current_height - 0.01 * y_range, current_height], 'k-', linewidth=1)
        ax.plot([other_pos, other_pos], [current_height - 0.01 * y_range, current_height], 'k-', linewidth=1)
        
        # Stars
        if p_corr < 0.001:
            p_text = '***'
        elif p_corr < 0.01:
            p_text = '**'
        elif p_corr < 0.05:
            p_text = '*'
        else:
            p_text = 'ns'
        
        ax.text((internal_pos + other_pos) / 2, current_height + 0.02 * y_range, p_text,
                ha='center', va='bottom', fontsize=12)
        annotation_count += 1

# Omnibus annotation
omnibus_text = f"{omnibus_method}: p < $10^{{-300}}$" if omnibus_p < 1e-300 else f"{omnibus_method}: p = {omnibus_p:.1e}"
ax.text(0.02, 0.98, omnibus_text, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', style='italic',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_ylim(ax.get_ylim()[0], annotation_height + annotation_count * height_increment + 0.1 * y_range)
ax.set_ylabel('Synergy (bits)', fontsize=22)
ax.tick_params(axis='x', rotation=30, labelsize=16)
ax.tick_params(axis='y', labelsize=18)

# Legend
legend_handles = [
    Line2D([0], [0], marker='$***$', color='k', linestyle='None', markersize=16),
    Line2D([0], [0], marker='$**$', color='k', linestyle='None', markersize=16),
    Line2D([0], [0], marker='$*$', color='k', linestyle='None', markersize=16),
]
legend_labels = ['p < 0.001 (FDR)', 'p < 0.01 (FDR)', 'p < 0.05 (FDR)']
ax.legend(legend_handles, legend_labels, title='Post-hoc significance', loc='best',
          fontsize=13, title_fontsize=13, bbox_to_anchor=(1.0, 0.7))

plt.tight_layout()
plt.savefig(FIG_PATH + 'fig4C_anova_posthoc.png', dpi=300, bbox_inches="tight")
plt.savefig(FIG_PATH + 'fig4C_anova_posthoc.pdf', bbox_inches="tight")
plt.show()

print("Figure saved.")

## 6. Save Full Statistical Results

In [ ]:
results = {
    'omnibus_method': omnibus_method,
    'omnibus_p': omnibus_p,
    'posthoc_table': posthoc,
    'effect_sizes': effect_df,
    'descriptive_stats': desc,
    'metrics_df': metrics_df,
}

if normality_ok:
    results['anova_table'] = aov
else:
    results['friedman_chi2'] = chi2
    results['kendalls_W'] = W

import pickle as pk
save_path = FIG_PATH + 'synergy_stats_results.pck'
with open(save_path, 'wb') as f:
    pk.dump(results, f)
print(f"Results saved to {save_path}")

# Print a concise paragraph for the paper
print("\n" + "=" * 70)
print("SUGGESTED TEXT FOR PAPER")
print("=" * 70)
internal_mean = metrics_df['internal'].mean()
internal_std = metrics_df['internal'].std()
print(f"""
We compared the mean pairwise synergy across five edge categories using a
{"repeated-measures ANOVA" if normality_ok else "Friedman test"} (N = {len(metrics_df)} subjects).
The omnibus test revealed a significant effect of edge category
({omnibus_method}: p = {omnibus_p:.1e}).
Post-hoc pairwise comparisons with FDR correction (Benjamini-Hochberg)
confirmed that internal edges (mean = {internal_mean:.3f} ± {internal_std:.3f} bits)
had significantly higher synergy than all other categories
(all p_FDR < 0.001). Effect sizes (Cohen's d) ranged from
{effect_df[effect_df['Group A'].str.contains('Internal') | effect_df['Group B'].str.contains('Internal')]["Cohen's d"].abs().min():.2f}
to {effect_df[effect_df['Group A'].str.contains('Internal') | effect_df['Group B'].str.contains('Internal')]["Cohen's d"].abs().max():.2f}.
""")